# Final Evaluation: Parameterized Scenario Matrix

This notebook evaluates trained DQN agents (and a heuristic baseline) on the
`ScalableEvalScenario` system — parameterized scenarios with **physics-guaranteed
feasibility**.

**Pipeline:**
1. Load or train agents on the full tutorial curriculum
2. Fine-tune on S32/S33 (urgency scenarios)
3. Evaluate across load sizes x yard fill levels
4. Compare DQN vs heuristic baseline
5. Generate thesis-ready figures

In [1]:
# ================================================================
# Cell 1: Imports & Configuration
# ================================================================
import os, sys, random, time, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

# Add project root to path (notebook is in notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

matplotlib.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})
warnings.filterwarnings('ignore', category=FutureWarning)

# Project imports
from simulation.training.curriculum_trainer import (
    create_env_factory, build_agent_config,
)
from simulation.rl.agent_registry import create_agent
from simulation.rl.multihead_dqn.config import Dims
from simulation.training.tutorial_runner import TutorialRunner
from simulation.training.scenarios import SCENARIO_BY_ID, TIER_DEFS
from simulation.training.eval import (
    ScalableEvalScenario, EvalParams,
    run_eval_matrix, default_eval_grid, default_eval_grid_with_trucks,
    summarize_eval, HeuristicAgent,
)

# ── User configuration ────────────────────────────────────────
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Agents to evaluate: list of (dqn_variant, cnn_backbone) tuples.
# Labels are auto-generated as "variant_backbone".
AGENTS_TO_EVAL = [
    ("baseline",      "baseline"),
    ("baseline",      "narrow_deep"),
    ("spectral_norm", "wider"),
    ("spectral_norm", "deeper"),
    ("munchausen",    "narrow_deep"),
    ("munchausen",    "wider"),
    ("munchausen",    "baseline"),
    ("noisynet",      "region_aware"),
    ("noisynet",      "narrow_deep"),
    ("noisynet",      "deeper"),
]

def agent_label(variant: str, backbone: str) -> str:
    """Generate a unique label for a (variant, backbone) combo."""
    return f"{variant}_{backbone}"

# Paths
CKPT_DIR = PROJECT_ROOT / 'runs' / 'tutorial_checkpoints'
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'final_eval'
FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation settings
EVAL_EPSILON = 0.05       # Exploration during eval
N_REPEATS = 5             # Repeats per parameter combo
MAX_RETRIES = 10          # Max total attempts per combo

# Warmup settings (per-combo adaptation before eval)
WARMUP_EPOCHS = 10        # Training episodes per combo before eval
WARMUP_EPSILON = 0.10     # Exploration during warmup
WARMUP_GRAD_STEPS = 4     # Gradient steps per warmup episode

# Training settings (if no checkpoint exists)
TRAIN_EPOCHS = 800
MASTERY_THRESHOLD = 0.9

# Fine-tuning settings
FINETUNE_EPOCHS = 50
FINETUNE_EPSILON = 0.05
FINETUNE_SCENARIOS = [32, 33]  # S32 (batch train loading), S33 (mixed urgency)

print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')
print(f'Output: {OUTPUT_DIR}')
print(f'Agents ({len(AGENTS_TO_EVAL)} combos):')
for v, b in AGENTS_TO_EVAL:
    print(f'  {agent_label(v, b):30s}  (DQN={v}, CNN={b})')
print(f'Warmup: {WARMUP_EPOCHS} epochs @ ε={WARMUP_EPSILON}, {WARMUP_GRAD_STEPS} grad steps/epoch')

Project root: /home/franko/CT-DRL-MA
Device: cuda
Output: /home/franko/CT-DRL-MA/runs/final_eval
Agents (10 combos):
  baseline_baseline               (DQN=baseline, CNN=baseline)
  baseline_narrow_deep            (DQN=baseline, CNN=narrow_deep)
  spectral_norm_wider             (DQN=spectral_norm, CNN=wider)
  spectral_norm_deeper            (DQN=spectral_norm, CNN=deeper)
  munchausen_narrow_deep          (DQN=munchausen, CNN=narrow_deep)
  munchausen_wider                (DQN=munchausen, CNN=wider)
  munchausen_baseline             (DQN=munchausen, CNN=baseline)
  noisynet_region_aware           (DQN=noisynet, CNN=region_aware)
  noisynet_narrow_deep            (DQN=noisynet, CNN=narrow_deep)
  noisynet_deeper                 (DQN=noisynet, CNN=deeper)
Warmup: 10 epochs @ ε=0.1, 4 grad steps/epoch


In [2]:
# ================================================================
# Cell 2: Utilities
# ================================================================

def seed_everything(seed=SEED):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def load_agent_from_checkpoint(agent, ckpt_path: str):
    """Load checkpoint into agent."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    agent.q_net.load_state_dict(ckpt['q_net'])
    agent.target_net.load_state_dict(ckpt['target_net'])
    agent.optimizer.load_state_dict(ckpt['optimizer'])
    agent.step_count = ckpt['step_count']
    print(f'  Loaded checkpoint (step {agent.step_count})')


# Environment and config factories
ROWS, BAYS, TIERS = 5, 58, 5
TRACKS, SPLIT_FACTOR = 7, 20

env_factory = create_env_factory(
    rows=ROWS, bays=BAYS, tiers=TIERS,
    tracks=TRACKS, split_factor=SPLIT_FACTOR,
    export_ratio=1.0, max_retries=2,
)

cfg = build_agent_config(
    rows=ROWS, bays=BAYS, tiers=TIERS,
    split_factor=SPLIT_FACTOR, tracks=TRACKS,
)

seed_everything(SEED)
print('Utilities ready.')

Utilities ready.


In [3]:
# ================================================================
# Cell 3: Agent Loading / Training
# ================================================================

agents: Dict[str, object] = {}

for variant, backbone in AGENTS_TO_EVAL:
    label = agent_label(variant, backbone)
    print(f'\n{"="*60}')
    print(f'Agent: {label} (DQN={variant}, CNN={backbone})')
    print(f'{"="*60}')

    agent = create_agent(variant, cfg, backbone_variant=backbone)

    # Try loading a checkpoint (keyed by combo label)
    ckpt_path = CKPT_DIR / f'{label}_best.pt'
    alt_path = CKPT_DIR / f'{label}.pt'

    loaded = False
    for path in [ckpt_path, alt_path]:
        if path.exists():
            print(f'Loading checkpoint: {path}')
            load_agent_from_checkpoint(agent, str(path))
            loaded = True
            break

    if not loaded:
        print(f'No checkpoint found. Training from scratch ({TRAIN_EPOCHS} epochs)...')
        runner = TutorialRunner(
            env_factory=env_factory,
            agent_or_config=agent,
            verbose=True,
        )
        runner.train_all(epochs=TRAIN_EPOCHS, mastery_threshold=MASTERY_THRESHOLD)

        # Save checkpoint
        CKPT_DIR.mkdir(parents=True, exist_ok=True)
        agent.save(str(ckpt_path))
        print(f'Saved checkpoint: {ckpt_path}')

    agents[label] = agent
    print(f'{label}: ready (step_count={agent.step_count})')

print(f'\nLoaded {len(agents)} DQN agent(s).')


Agent: baseline_baseline (DQN=baseline, CNN=baseline)
No checkpoint found. Training from scratch (800 epochs)...

  Tutorial tiers (14 total):
    Tier 0: Primitives [S1, S2, S3, S4]
    Tier 1: Two-action chains [S5, S6, S26, S27]
    Tier 2: Full chains [S7, S8]
    Tier 3: Restack + load [S9, S10]
    Tier 4: Random generalize [S11, S12]
    Tier 5: Multi-step hard [S13, S14]
    Tier 6: Terminal truck [S25, S22, S23, S24]
    Tier 7: Multi-vehicle [S15, S16]
    Tier 8: Bidirectional train [S17]
    Tier 9: Concurrent ops [S18, S19]
    Tier 10: Full complexity [S20, S21]
    Tier 11: Full train ops [S28, S29]
    Tier 12: Cross-modal orchestration [S30, S31]
    Tier 13: Operational readiness [S32, S33]



/home/franko/.cache/pypoetry/virtualenvs/ct-drl-ma-7RJlkqYG-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator KernelDensity from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



  Epoch 5/800 — Tier 0: Primitives
    ► Tier 0: Primitives
      [====================] 100.0% OK S1: park_truck
      [====================] 100.0% OK S2: train_import
      [--------------------]  0.0%    S3: yard_to_truck
      [--------------------]  0.0%    S4: yard_to_train
    🔒 Tier 1: Two-action chains
    🔒 Tier 2: Full chains
    🔒 Tier 3: Restack + load
    🔒 Tier 4: Random generalize
    🔒 Tier 5: Multi-step hard
    🔒 Tier 6: Terminal truck
    🔒 Tier 7: Multi-vehicle
    🔒 Tier 8: Bidirectional train
    🔒 Tier 9: Concurrent ops
    🔒 Tier 10: Full complexity
    🔒 Tier 11: Full train ops
    🔒 Tier 12: Cross-modal orchestration
    🔒 Tier 13: Operational readiness
    Average (active): 50.0% | Mastered: 2/33 | Graduated: 0 | ε=0.858

  Epoch 10/800 — Tier 0: Primitives
    ► Tier 0: Primitives
      [====================] 100.0% OK S1: park_truck
      [====================] 100.0% OK S2: train_import
      [--------------------]  0.0%    S3: yard_to_truck
      [==--

KeyboardInterrupt: 

In [ ]:
# ================================================================
# Cell 4: Fine-tuning on S32/S33 (Urgency Scenarios)
# ================================================================

for label, agent in agents.items():
    if not getattr(agent, 'is_trainable', True):
        continue  # skip heuristic

    print(f'\nFine-tuning {label} on S{FINETUNE_SCENARIOS}...')

    # Set exploration
    agent.epsilon_override = FINETUNE_EPSILON
    if hasattr(agent, 'set_noise_scale'):
        agent.set_noise_scale(0.1)

    runner = TutorialRunner(
        env_factory=env_factory,
        agent_or_config=agent,
        verbose=False,
    )

    for epoch in range(FINETUNE_EPOCHS):
        epoch_rewards = []
        for sid in FINETUNE_SCENARIOS:
            sc = SCENARIO_BY_ID[sid]
            result = runner.run_scenario(sc)
            epoch_rewards.append(result.total_reward)

            # Gradient steps
            if hasattr(agent, 'replay') and hasattr(agent.replay, 'is_ready'):
                if agent.replay.is_ready(agent.cfg.training.batch_size):
                    for _ in range(4):
                        agent.optimize()

        if (epoch + 1) % 10 == 0:
            # Greedy validation pass
            old_eps = agent.epsilon_override
            agent.epsilon_override = 0.0
            val_results = []
            for sid in FINETUNE_SCENARIOS:
                sc = SCENARIO_BY_ID[sid]
                r = runner.run_scenario(sc)
                val_results.append(('PASS' if r.passed else 'FAIL', r.total_reward))
            agent.epsilon_override = old_eps

            val_str = '  '.join(
                f'S{sid}={tag} R={rew:+.1f}'
                for sid, (tag, rew) in zip(FINETUNE_SCENARIOS, val_results)
            )
            print(f'  Epoch {epoch+1:3d}/{FINETUNE_EPOCHS}: {val_str}')

    # Clear overrides
    agent.epsilon_override = None
    if hasattr(agent, 'set_noise_scale'):
        agent.set_noise_scale(1.0)

print('\nFine-tuning complete.')

In [ ]:
# ================================================================
# Cell 5: Heuristic Baseline
# ================================================================

heuristic = HeuristicAgent(cfg.unified)
agents['heuristic'] = heuristic

print(f'Heuristic baseline created.')
print(f'Total agents for evaluation: {list(agents.keys())}')

In [ ]:
# ================================================================
# Cell 6: Evaluation Matrix (with per-combo warmup)
# ================================================================

grid = default_eval_grid()  # 15 combos: 5 loads x 3 fills

print(f'Evaluation grid: {len(grid)} parameter combos')
print(f'Warmup: {WARMUP_EPOCHS} epochs @ ε={WARMUP_EPSILON} per combo (DQN only)')
print(f'Eval: {N_REPEATS} repeats @ ε={EVAL_EPSILON} (max {MAX_RETRIES} with retries)')
print()

# Show grid
for i, p in enumerate(grid):
    sc = ScalableEvalScenario(p)
    print(
        f'  [{i+1:2d}] {p.label:40s}  '
        f'trains={sc._n_trains}  '
        f'window={sc._feasible_seconds/60:.0f}min  '
        f'max_steps={sc.max_steps}'
    )

all_results: Dict[str, pd.DataFrame] = {}

for label, agent in agents.items():
    print(f'\n{"="*60}')
    print(f'Evaluating: {label}')
    print(f'{"="*60}')

    # Skip warmup for non-trainable agents (heuristic)
    is_trainable = getattr(agent, 'is_trainable', True)
    agent_warmup = WARMUP_EPOCHS if is_trainable else 0

    # Keep NoisyNet noise active (learned exploration)
    # Only zero noise for heuristic or non-noisy agents
    if hasattr(agent, 'set_noise_scale'):
        agent.set_noise_scale(1.0)

    # Initial run with warmup
    t0 = time.time()
    df = run_eval_matrix(
        agent, env_factory, grid,
        n_repeats=N_REPEATS,
        warmup_epochs=agent_warmup,
        warmup_epsilon=WARMUP_EPSILON,
        warmup_grad_steps=WARMUP_GRAD_STEPS,
        eval_epsilon=EVAL_EPSILON,
        verbose=True,
    )
    elapsed = time.time() - t0
    print(f'  Initial run: {len(df)} runs in {elapsed:.1f}s')

    # Retry logic: for combos where not all runs passed,
    # run additional attempts up to MAX_RETRIES total
    # (retries use no extra warmup — agent already adapted)
    group_cols = ['n_imports', 'n_exports', 'yard_fill_pct']
    for _, grp in df.groupby(group_cols):
        if grp['passed'].all():
            continue  # All passed — no retries needed

        n_done = len(grp)
        if n_done >= MAX_RETRIES:
            continue  # Already at max

        # Extract the EvalParams for this combo
        row0 = grp.iloc[0]
        retry_params = EvalParams(
            n_imports=int(row0['n_imports']),
            n_exports=int(row0['n_exports']),
            n_delivery_trucks=int(row0.get('n_delivery_trucks', 0)),
            n_pickup_trucks=int(row0.get('n_pickup_trucks', 0)),
            yard_fill_pct=float(row0['yard_fill_pct']),
            seed=int(row0['seed']),
        )

        n_extra = MAX_RETRIES - n_done
        # Use offset seeds so we don't reuse the same layouts
        retry_grid = [
            EvalParams(
                n_imports=retry_params.n_imports,
                n_exports=retry_params.n_exports,
                n_delivery_trucks=retry_params.n_delivery_trucks,
                n_pickup_trucks=retry_params.n_pickup_trucks,
                yard_fill_pct=retry_params.yard_fill_pct,
                seed=retry_params.seed + (n_done + i) * 1000,
            )
            for i in range(n_extra)
        ]
        extra_df = run_eval_matrix(
            agent, env_factory, retry_grid,
            n_repeats=1,
            warmup_epochs=0,  # no extra warmup for retries
            eval_epsilon=EVAL_EPSILON,
            verbose=False,
        )
        df = pd.concat([df, extra_df], ignore_index=True)
        n_passed = df[
            (df['n_imports'] == retry_params.n_imports) &
            (df['n_exports'] == retry_params.n_exports) &
            (df['yard_fill_pct'] == retry_params.yard_fill_pct)
        ]['passed'].sum()
        print(
            f'  Retried {retry_params.label}: '
            f'{n_passed}/{n_done + n_extra} passed'
        )

    all_results[label] = df
    print(f'  Total runs for {label}: {len(df)}')

    # Clear epsilon
    if hasattr(agent, 'clear_epsilon_override'):
        agent.clear_epsilon_override()

print(f'\nEvaluation complete for {len(all_results)} agents.')

In [ ]:
# ================================================================
# Cell 7: Summary Statistics
# ================================================================

summaries: Dict[str, pd.DataFrame] = {}

for label, df in all_results.items():
    print(f'\n{"="*60}')
    print(f'Summary: {label}')
    print(f'{"="*60}')

    summary = summarize_eval(df)
    summaries[label] = summary

    # Overall stats
    n_runs = len(df)
    n_passed = df['passed'].sum()
    print(f'  Runs: {n_runs}, Passed: {n_passed} ({n_passed/n_runs:.0%})')
    print(f'  Avg completion: {df["completion_pct"].mean():.1%} +/- {df["completion_pct"].std():.1%}')
    print(f'  Avg moves/container: {df["moves_per_container"].mean():.2f}')
    print(f'  Avg reshuffle ratio: {df["reshuffle_ratio"].mean():.2%}')
    print(f'  Avg reward: {df["total_reward"].mean():+.1f}')
    print()

    # Per-combo breakdown
    display_cols = [
        'n_imports', 'n_exports', 'yard_fill_pct', 'total_containers',
        'n_runs', 'pass_rate', 'completion_mean', 'completion_std',
        'moves_per_container_mean', 'reshuffle_ratio_mean', 'reward_mean',
    ]
    avail_cols = [c for c in display_cols if c in summary.columns]
    print(summary[avail_cols].to_string(index=False, float_format='%.3f'))

    # Sub-task breakdown (if available)
    sub_task_cols = [c for c in df.columns if c.endswith('_pct') and c != 'completion_pct']
    if sub_task_cols:
        print(f'\n  Sub-task averages:')
        for col in sub_task_cols:
            vals = df[col].dropna()
            if len(vals) > 0:
                print(f'    {col:30s} {vals.mean():.1%} +/- {vals.std():.1%}')

In [ ]:
# ================================================================
# Cell 8: Thesis Figures
# ================================================================

# Colour palette
AGENT_COLORS = {
    'kitchen_sink': '#2196F3',
    'heuristic': '#FF9800',
}

FILL_LABELS = {0.0: '0%', 0.25: '25%', 0.50: '50%'}
LOAD_SIZES = [10, 20, 40, 60, 80]


def _agent_color(label: str) -> str:
    return AGENT_COLORS.get(label, '#607D8B')


# ────────────────────────────────────────────────────────────────
# Figure 1: Pass Rate Heatmap
# ────────────────────────────────────────────────────────────────

def plot_pass_rate_heatmap(all_results, summaries):
    agent_labels = list(all_results.keys())
    n_agents = len(agent_labels)
    fig, axes = plt.subplots(1, n_agents, figsize=(5 * n_agents, 3.5),
                             squeeze=False)

    for idx, label in enumerate(agent_labels):
        ax = axes[0, idx]
        df = all_results[label]

        # Build heatmap matrix: rows=fill, cols=load
        fills = sorted(df['yard_fill_pct'].unique())
        loads = sorted(df['total_containers'].unique())

        matrix = np.full((len(fills), len(loads)), np.nan)
        for i, fill in enumerate(fills):
            for j, load in enumerate(loads):
                subset = df[
                    (df['yard_fill_pct'] == fill) &
                    (df['total_containers'] == load)
                ]
                if len(subset) > 0:
                    matrix[i, j] = subset['passed'].mean()

        im = ax.imshow(
            matrix, cmap='RdYlGn', vmin=0, vmax=1,
            aspect='auto', origin='lower',
        )
        ax.set_xticks(range(len(loads)))
        ax.set_xticklabels([str(int(l)) for l in loads])
        ax.set_yticks(range(len(fills)))
        ax.set_yticklabels([FILL_LABELS.get(f, f'{f:.0%}') for f in fills])
        ax.set_xlabel('Total Containers')
        ax.set_ylabel('Yard Fill')
        ax.set_title(label.replace('_', ' ').title())

        # Annotate cells
        for i in range(len(fills)):
            for j in range(len(loads)):
                val = matrix[i, j]
                if not np.isnan(val):
                    color = 'white' if val < 0.5 else 'black'
                    ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                            fontsize=9, fontweight='bold', color=color)

    fig.suptitle('Pass Rate by Load Size and Yard Fill', fontsize=13, y=1.02)
    fig.colorbar(im, ax=axes.ravel().tolist(), label='Pass Rate',
                 shrink=0.8, pad=0.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'pass_rate_heatmap.png', bbox_inches='tight')
    plt.show()


plot_pass_rate_heatmap(all_results, summaries)


# ────────────────────────────────────────────────────────────────
# Figure 2: Completion % Scaling Curve
# ────────────────────────────────────────────────────────────────

def plot_completion_scaling(all_results):
    fig, ax = plt.subplots(figsize=(8, 5))

    for label, df in all_results.items():
        color = _agent_color(label)
        for fill in sorted(df['yard_fill_pct'].unique()):
            sub = df[df['yard_fill_pct'] == fill]
            grp = sub.groupby('total_containers')['completion_pct'].agg(['mean', 'std'])
            grp = grp.sort_index()

            linestyle = {0.0: '-', 0.25: '--', 0.50: ':'}. get(fill, '-.')
            fill_lbl = FILL_LABELS.get(fill, f'{fill:.0%}')

            ax.plot(
                grp.index, grp['mean'],
                marker='o', markersize=4,
                linestyle=linestyle, color=color,
                label=f'{label} (fill={fill_lbl})',
            )
            ax.fill_between(
                grp.index,
                (grp['mean'] - grp['std']).clip(0, 1),
                (grp['mean'] + grp['std']).clip(0, 1),
                alpha=0.1, color=color,
            )

    ax.set_xlabel('Total Containers')
    ax.set_ylabel('Completion %')
    ax.set_title('Completion Rate vs. Scenario Size')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5, label='Perfect')
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'completion_scaling.png', bbox_inches='tight')
    plt.show()


plot_completion_scaling(all_results)


# ────────────────────────────────────────────────────────────────
# Figure 3: Move Distribution Stacked Bar
# ────────────────────────────────────────────────────────────────

def plot_move_distribution(all_results):
    move_types = [
        'YARD_TO_TRAIN', 'TRAIN_TO_YARD', 'PARK_TRUCK',
        'TRUCK_TO_YARD', 'YARD_TO_TRUCK', 'YARD_TO_YARD',
    ]
    move_colors = {
        'YARD_TO_TRAIN': '#4CAF50',
        'TRAIN_TO_YARD': '#2196F3',
        'PARK_TRUCK': '#FF9800',
        'TRUCK_TO_YARD': '#9C27B0',
        'YARD_TO_TRUCK': '#E91E63',
        'YARD_TO_YARD': '#F44336',
    }

    agent_labels = list(all_results.keys())
    n_agents = len(agent_labels)

    # Use fill=0% for cleaner comparison
    fig, axes = plt.subplots(1, n_agents, figsize=(5 * n_agents, 4.5),
                             squeeze=False, sharey=True)

    for idx, label in enumerate(agent_labels):
        ax = axes[0, idx]
        df = all_results[label]
        sub = df[df['yard_fill_pct'] == 0.0]
        if len(sub) == 0:
            sub = df  # fallback if no 0% fill

        loads = sorted(sub['total_containers'].unique())
        x = np.arange(len(loads))
        width = 0.6

        bottoms = np.zeros(len(loads))
        for mt in move_types:
            col = f'moves_{mt}'
            if col not in sub.columns:
                continue
            means = [sub[sub['total_containers'] == l][col].mean()
                     for l in loads]
            means = [m if not np.isnan(m) else 0 for m in means]
            ax.bar(x, means, width, bottom=bottoms,
                   label=mt.replace('_', ' ').title(),
                   color=move_colors.get(mt, '#999'))
            bottoms += np.array(means)

        ax.set_xticks(x)
        ax.set_xticklabels([str(int(l)) for l in loads])
        ax.set_xlabel('Total Containers')
        ax.set_ylabel('Avg. Move Count')
        ax.set_title(f'{label.replace("_", " ").title()} (fill=0%)')

    axes[0, 0].legend(fontsize=7, loc='upper left')
    fig.suptitle('Move Type Distribution by Scenario Size', fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'move_distribution.png', bbox_inches='tight')
    plt.show()


plot_move_distribution(all_results)


# ────────────────────────────────────────────────────────────────
# Figure 4: Efficiency Metrics (2x2)
# ────────────────────────────────────────────────────────────────

def plot_efficiency_metrics(all_results):
    metrics = [
        ('moves_per_container', 'Moves per Container'),
        ('reshuffle_ratio', 'Reshuffle Ratio'),
        ('completion_pct', 'Completion %'),
        ('total_reward', 'Total Reward'),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.ravel()

    for ax_idx, (metric, title) in enumerate(metrics):
        ax = axes[ax_idx]
        for label, df in all_results.items():
            color = _agent_color(label)
            # Average across all fills for simplicity
            grp = df.groupby('total_containers')[metric].agg(['mean', 'std']).sort_index()
            ax.plot(
                grp.index, grp['mean'],
                marker='o', markersize=5, color=color,
                label=label.replace('_', ' ').title(),
            )
            ax.fill_between(
                grp.index,
                grp['mean'] - grp['std'],
                grp['mean'] + grp['std'],
                alpha=0.15, color=color,
            )

        ax.set_xlabel('Total Containers')
        ax.set_ylabel(title)
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle('Efficiency Metrics vs. Scenario Size', fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'efficiency_metrics.png', bbox_inches='tight')
    plt.show()


plot_efficiency_metrics(all_results)


# ────────────────────────────────────────────────────────────────
# Figure 5: Sub-Task Breakdown (Hardest Scenario)
# ────────────────────────────────────────────────────────────────

def plot_subtask_breakdown(all_results):
    """Grouped bar chart of per-subtask completion at the hardest scenario."""
    sub_tasks = ['imports_unloaded', 'exports_loaded']
    agent_labels = list(all_results.keys())

    # Find hardest scenario (largest load, highest fill)
    ref_df = list(all_results.values())[0]
    max_load = ref_df['total_containers'].max()
    max_fill = ref_df['yard_fill_pct'].max()

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(sub_tasks))
    width = 0.35
    offset = np.linspace(-width/2, width/2, len(agent_labels))

    for i, label in enumerate(agent_labels):
        df = all_results[label]
        sub = df[
            (df['total_containers'] == max_load) &
            (df['yard_fill_pct'] == max_fill)
        ]

        vals = []
        for st in sub_tasks:
            col = f'{st}_pct'
            if col in sub.columns and len(sub[col].dropna()) > 0:
                vals.append(sub[col].mean())
            else:
                vals.append(0.0)

        ax.bar(
            x + offset[i], vals, width / len(agent_labels),
            label=label.replace('_', ' ').title(),
            color=_agent_color(label), alpha=0.85,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('_', ' ').title() for s in sub_tasks])
    ax.set_ylabel('Completion %')
    ax.set_title(
        f'Sub-Task Completion (Hardest: {int(max_load)} containers, '
        f'{FILL_LABELS.get(max_fill, f"{max_fill:.0%}")} fill)'
    )
    ax.set_ylim(0, 1.1)
    ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'subtask_breakdown.png', bbox_inches='tight')
    plt.show()


plot_subtask_breakdown(all_results)

In [ ]:
# ================================================================
# Cell 9: Export Results
# ================================================================

for label, df in all_results.items():
    path = OUTPUT_DIR / f'raw_{label}.csv'
    df.to_csv(path, index=False)
    print(f'Saved raw results: {path}')

for label, summary in summaries.items():
    path = OUTPUT_DIR / f'summary_{label}.csv'
    summary.to_csv(path, index=False)
    print(f'Saved summary: {path}')

# Combined comparison table
comparison_rows = []
for label, df in all_results.items():
    comparison_rows.append({
        'agent': label,
        'total_runs': len(df),
        'pass_rate': df['passed'].mean(),
        'avg_completion': df['completion_pct'].mean(),
        'avg_moves_per_container': df['moves_per_container'].mean(),
        'avg_reshuffle_ratio': df['reshuffle_ratio'].mean(),
        'avg_reward': df['total_reward'].mean(),
    })

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUTPUT_DIR / 'agent_comparison.csv', index=False)
print(f'\nSaved comparison: {OUTPUT_DIR / "agent_comparison.csv"}')
print('\n--- Agent Comparison ---')
print(comparison.to_string(index=False, float_format='%.3f'))

print(f'\nAll figures saved to: {FIG_DIR}')
print('Done!')